# Enforcing Global Policies

## Key Concepts

- **Global Contract**: A data contract as code that gets defined at the mesh level.
- **Enforce a contract as a Policy**: Activate a global contract on a set of data products to enforce a contract.

## Prerequisites

1. Ensure that the [credit-card-tx data product](./credit_card_tx/spec.py) source is available in this jupyter instance
2. Ensure that [pii_compliance](./contracts/pii_compliance.py) is available in this jupyter instance

In [ ]:
%%capture

!curl -LsSf https://astral.sh/uv/install.sh | sh

In [ ]:
!nxd undeploy credit-card-tx-demo

In [ ]:
%%bash
rm -rf .venv
uv venv --python 3.11
uv pip install --no-cache --link-mode=copy --upgrade --index-url=http://registry.demo.trynxd.com/index --extra-index-url=https://pypi.org/simple -r credit_card_tx/requirements.txt

## Setup

Let's launch a our data product for this tutorial.

In [ ]:
%%bash

source .venv/bin/activate
nxd launch --dir credit_card_tx --skip-status-polling

## Step 1: Global Contract
Take a look at the [pii_compliance](./contracts/pii_compliance.py) as a sample of contract that you want to enforce across data products.

In [ ]:
# Register a contract with code file
!nxd contracts create \
  --name credit_card_tx_pii_compliance_contract \
  --code ./contracts/pii_compliance.py \
  --description "PII Compliance Contract for credit-card-tx - Customers"

Test and see it in the list of global contracts:

In [ ]:
# List all registered contracts
!nxd contracts list

In [ ]:
# Get contract details
!nxd contracts describe credit_card_tx_pii_compliance_contract

## Activate the policy

This step applies the policy to the **fraud-detection** domain and targets any data products containing a model with an `email` attribute

In [ ]:
# Activate the policy
!nxd policies activate \
  --name credit-card-tx-pii-compliance \
  --description "PII Compliance Contract for credit-card-tx - Customers" \
  --policy DataQualityCompliance \
  --promise-enforced \
  --validation-url credit_card_tx_pii_compliance_contract \
  --output-ports at-least-one \
  --filter '.domain == "fraud-detection" and (.models.models[].attributes | to_entries[] | select(.key == "email") | any)' \
  --env demo \
  --consequence STOP

Test to see the result:

In [ ]:
# Get policy activation details
!nxd policies ls data-product policies --name "credit-card-tx" --env demo

## Activate data product contract

The data product is currently failing with 

> ERROR: Data quality compliance policy violation: no promises contracts found to validate - policy requires contracts to be defined


To fix this, modify the [credit_card_tx/spec.py](credit_card_tx/spec.py) to promise that it will adhere to the pii_compliance contract.

In [ ]:
%%bash
source .venv/bin/activate

# Relaunch data product
nxd launch --dir=credit_card_tx

## Clean up

In [ ]:
# Deactivate the policy
!nxd policies deactivate --name credit-card-tx-pii-compliance

In [ ]:
# Delete the policy
!nxd policies delete --name credit-card-tx-pii-compliance

In [ ]:
# Undeploy data product
!nxd undeploy credit-card-tx-demo

In [ ]:
# Delete the contract
!nxd contracts delete credit_card_tx_pii_compliance_contract -y